# Задача 1. Tracer

Реализовать декоратор, который показывает стек вызовов рекурсивных функций.

На каждом шаге должен быть виден:

- вход в рекурсию (вызов функции);
- отступ, соответствующий глубине стека;
- возврат из рекурсии с результатом.

## Идея

Декоратор заменяет функцию обёрткой. Перед вызовом она печатает вход и увеличивает глубину. Когда функция заканчивает работу, глубина уменьшается, а обёртка печатает возвращённое значение. `finally` восстанавливает глубину даже при ошибке. Каждый уровень рекурсии получает свой отступ из двух пробелов.

In [1]:
def tracer(function):
    depth = 0

    def wrapper(*args, **kwargs):
        nonlocal depth
        parts = [repr(value) for value in args]
        parts += [f"{name}={value!r}" for name, value in kwargs.items()]
        arguments = ", ".join(parts)
        indent = "  " * depth

        print(f"{indent}→ {function.__name__}({arguments})")
        depth += 1
        try:
            result = function(*args, **kwargs)
        finally:
            depth -= 1
        print(f"{indent}← {function.__name__} = {result!r}")
        return result

    return wrapper

## Пример работы

Возьмём факториал `3`. Сначала стек растёт: `factorial(3) → factorial(2) → factorial(1)`. Затем вызовы возвращают `1`, `2` и `6` в обратном порядке. У каждого вызова ниже первого отступ больше на два пробела.

In [2]:
@tracer
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

factorial(3)

→ factorial(3)
  → factorial(2)
    → factorial(1)
    ← factorial = 1
  ← factorial = 2
← factorial = 6


6

## Сложность

Пусть выполнено `m` вызовов, а максимальная глубина равна `d`. Обёртка хранит `O(d)` активных вызовов. Если не считать печать, она добавляет постоянную работу на вызов — всего `O(m)`. Сама печать зависит от длины аргументов, результатов и отступов: для цепочки из `m` рекурсивных вызовов только отступы занимают `O(m²)` символов. Поэтому в программе с трассировкой вывод может быть заметно дороже самого вычисления.

## Тесты

Проверяем базовый случай, несколько уровней рекурсии, возвращаемое значение, именованные аргументы и то, что следующий независимый вызов снова начинается без отступа.

In [3]:
from contextlib import redirect_stdout
from io import StringIO

output = StringIO()
with redirect_stdout(output):
    assert factorial(4) == 24
    assert factorial(0) == 1

lines = output.getvalue().splitlines()
assert lines[0] == "→ factorial(4)"
assert lines[1] == "  → factorial(3)"
assert "    → factorial(2)" in lines
assert "  ← factorial = 6" in lines
assert "← factorial = 24" in lines
assert lines[-2] == "→ factorial(0)"
assert lines[-1] == "← factorial = 1"

@tracer
def add(a, b=0):
    return a + b

output = StringIO()
with redirect_stdout(output):
    assert add(2, b=3) == 5
assert output.getvalue().splitlines() == [
    "→ add(2, b=3)", "← add = 5"
]

print("Задача 1: все тесты пройдены")

Задача 1: все тесты пройдены
